# W6 — Inference Rebuild: One Model, Several Calculations

**October 6, 2026 · CPU · no upload, API key or GPU.** Save a copy in Drive. Run from top to bottom. If setup requests a restart, restart once and run all again.

We keep observations (1,0,1) fixed and infer their hidden states. Predict before running. Unfinished exercises do not block later demonstrations. The final section opens an interactive app and asks you to modify one diagnostic.

[Study notes](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/course/notion/w6/w6-inference-rebuild.md) · [Repair note](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/course/notion/w6/w6-repair.md) · [Worksheet](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/course/notion/w6/w6-experiment-record.md) · [Complete API guide](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/course/notion/w6/w6-code-guide.md)

Setup uses pinned package versions and checked source hashes. A temporary Gradio share URL lasts only while your runtime is active; keep this notebook link as the permanent entry point.


In [ ]:
import sys, subprocess, importlib.metadata, hashlib, urllib.request, time
from pathlib import Path

PINS = {"numpy":"2.2.6", "matplotlib":"3.10.3", "scipy":"1.16.3",
        "pandas":"2.3.3", "arviz":"0.22.0", "gradio":"6.27.0"}
missing = []
for package, version in PINS.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(f"{package}=={version}")
restart = [p for p,v in PINS.items() if p in sys.modules and getattr(sys.modules[p], "__version__", v) != v]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
if restart:
    raise RuntimeError("Packages updated. Use Runtime > Restart session, then Run all. Loaded old packages: "+", ".join(restart))

CODE_REF = "2026-fall-w6"
EXPECTED = {'advml_pgm.py': '97c68026a1024694ec97a632449aa6ada97d18fe0a8fb94212d260818fbf0a28', 'advml_rebuild.py': 'd10328cc8fc3e581fdc30f369c0c44835159b760e3e0ce9f919a3023d5ba7562', 'advml_rebuild_app.py': 'c536deb468c718d8210516984419c55070813d5c3cc5058d5340bff4ddeb4efa'}
local = Path.cwd() / "src"
local_ok = all((local/name).is_file() and hashlib.sha256((local/name).read_bytes()).hexdigest()==digest
               for name,digest in EXPECTED.items())
support = local if local_ok else Path.cwd() / "w6_support" / CODE_REF
support.mkdir(parents=True, exist_ok=True)
downloaded = []
for name,digest in EXPECTED.items():
    path = support/name
    if path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest()==digest:
        continue
    url = f"https://raw.githubusercontent.com/lunalab-ai/advML/{CODE_REF}/src/{name}"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=40) as response:
                payload = response.read()
            break
        except (OSError, TimeoutError):
            if attempt == 2:
                raise
            time.sleep(attempt+1)
    if hashlib.sha256(payload).hexdigest()!=digest:
        raise RuntimeError(f"Source hash mismatch: {name}. Do not run unverified code.")
    temporary = path.with_suffix(".download")
    temporary.write_bytes(payload)
    temporary.replace(path)
    downloaded.append(name)
sys.path.insert(0, str(support.resolve()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from IPython.display import display
print({"Python":sys.version.split()[0], "code_ref":CODE_REF, "downloaded":downloaded,
       "cache_or_reviewed_local":not downloaded, "versions":PINS})


## 1. The table is our shared reference

[SensorModel: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild.py#L13) · [SensorModel.pairwise: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild.py#L34) · [SensorModel.log_joint: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild.py#L46) · [exact_posterior: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild.py#L61) · [tree_sum_product: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_pgm.py#L149) · [distribution_summary: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild.py#L137)

`SensorModel()` fixes the observations, transition stay probability and sensor accuracy. `exact_posterior` returns paths `(8,3)`, normalized probabilities `(8,)`, marginal tables `(3,2)` and log evidence. `tree_sum_product` accepts the model's pairwise representation and returns the same marginal table using reusable messages. No parameters are fitted.


In [ ]:
from advml_rebuild import (SensorModel, exact_posterior, exact_filter, mean_field,
    sample_paths, particle_filter, distribution_summary)
from advml_pgm import tree_sum_product
from advml_rebuild_app import static_experiment, sequential_experiment, create_app
model = SensorModel()
ref = exact_posterior(model)
states, posterior = ref['states'], ref['probabilities']
table = pd.DataFrame(states, columns=['z1','z2','z3'])
table['joint_weight'] = np.exp(ref['log_joint'])
table['posterior'] = posterior
display(table)
print('Evidence:', np.exp(ref['log_z']))
print('Our main query:', distribution_summary(states, posterior))
messages = tree_sum_product(model.pairwise())
print('Largest exact/message discrepancy:', np.max(abs(messages['marginals']-ref['marginals'])))


### E1 — Marginal or conditional?

List the paths whose middle state is one and add their posterior probabilities. Then compare with the exact filter at time 2. Explain why the two answers can differ. Write your prediction before running your calculation.


In [ ]:
# Your calculation / explanation can go here. Later demonstrations are independent.
e1_response = ""


## 2. Restricting q and optimizing q are different decisions

[mean_field: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild.py#L104) · [distribution_summary: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild.py#L137)

`mean_field(model, initial, sweeps)` returns Bernoulli factors, eight path probabilities, an ELBO trace including the initial point, and reverse KL. It performs sequential coordinate updates and may reach different stationary points. A small update is not proof of a global optimum. The diagnostic covariance checks dependence, not only the event mean.


In [ ]:
vi_rows = []
for stay in [.5,.8,.95]:
    current = SensorModel(stay=stay)
    truth = exact_posterior(current)
    for initial in ([.1,.5,.9] if stay==.95 else [.5]):
        approx = mean_field(current, initial=initial)
        truth_summary = distribution_summary(truth['states'],truth['probabilities'])
        approx_summary = distribution_summary(truth['states'],approx['probabilities'])
        vi_rows.append(dict(stay=stay,initial=initial,exact=truth_summary['probability_last_one'],
            estimate=approx_summary['probability_last_one'],ELBO=approx['elbo'][-1],KL=approx['reverse_kl'],
            covariance_error=abs(truth_summary['covariance_first_last']-approx_summary['covariance_first_last'])))
vi_frame = pd.DataFrame(vi_rows)
display(vi_frame)
fig, ax = plt.subplots(figsize=(7,3))
for initial in [.1,.5,.9]:
    result = mean_field(SensorModel(stay=.95),initial)
    ax.plot(result['elbo'],label=f'initial={initial}')
ax.axhline(exact_posterior(SensorModel(stay=.95))['log_z'],ls='--',color='black',label='log evidence')
ax.set(xlabel='Coordinate sweeps',ylabel='ELBO'); ax.legend(); plt.show()


### E2 — Explain two kinds of error

At stay=.5, .8 and .95, what changes in the model? Identify evidence for a family restriction and evidence for an optimization problem. Can you claim the best observed run is globally optimal?


## 3. More computation, same data

[sample_paths: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild.py#L174)

`sample_paths(model, method, n, seed)` returns retained paths `(n,3)`, normalized weights `(n,)` and an eight-bin histogram. MC uses the exact oracle. IS uses a Bernoulli proposal and weights; MH keeps rejected visits; Gibbs retains one path after each full coordinate sweep. Chains default to 100 burn-in steps/sweeps. Weight ESS is returned only for IS.


In [ ]:
exact_probability = posterior @ states[:,-1]
sampling_rows = []
for method in ['MC','IS','MH','Gibbs']:
    result = sample_paths(model,method,n=400,seed=19)
    estimate = result['probabilities'] @ states[:,-1]
    sampling_rows.append(dict(method=method,estimate=estimate,error=abs(estimate-exact_probability),
        weight_ess=result['weight_ess'],move_fraction=result['move_fraction']))
display(pd.DataFrame(sampling_rows))
records = []
for n in [25,100,400]:
    estimates = np.array([sample_paths(model,'MC',n,seed)['probabilities'] @ states[:,-1] for seed in range(40)])
    records.append(dict(n=n,rmse=np.sqrt(np.mean((estimates-exact_probability)**2)),
        iid_mcse=np.sqrt(exact_probability*(1-exact_probability)/n),
        posterior_sd=np.sqrt(exact_probability*(1-exact_probability))))
mc_frame=pd.DataFrame(records); display(mc_frame)


### E3 — Interpret uncertainty and replication

Explain which column should shrink with budget and which should remain fixed. Must every individual seed improve? Why must MH not reuse the iid MCSE formula with its raw sample count?


## 4. Optional: new observation, new target

[exact_filter: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild.py#L71) · [particle_filter: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild.py#L223) · [sequential_experiment: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild_app.py#L59)

The exact filter returns prefix-specific probabilities. `particle_filter` returns these estimates plus `(T,n)` arrays of particles, weights and ancestor indices. It summarizes weighted particles before the next resampling step. No future observation is used.


In [ ]:
filter_plot, filter_table, filter_csv = sequential_experiment(n=400,seed=19,stay=.8)
display(filter_plot); display(filter_table)
print('Exported record:',filter_csv)


### E5 — Match the reference

Try 30 and 400 particles across several seeds. Explain why time-2 smoothing is the wrong reference for this online filter. Why do repeated zeros and ones not reveal the number of distinct ancestors?


## 5. Extend and launch the Inference Workbench

[static_experiment: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild_app.py#L11) · [sequential_experiment: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild_app.py#L59) · [create_app: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w6/src/advml_rebuild_app.py#L81)

`static_experiment` returns a Matplotlib Figure, a metrics dictionary and a CSV path. `create_app(extra=...)` builds Gradio Blocks without launching a server. The optional callback receives a copy of the metrics and returns only new diagnostic fields. It cannot replace the model's original metrics.

### E4 — Add one diagnostic

Edit `student_extra(metrics)` below. Return a dictionary with `covariance_error`, the absolute difference between `exact_covariance` and `estimate_covariance`. Launch the app, compare Exact and VI, and check that the field appears in the displayed JSON and exported CSV. The empty default keeps the app runnable.


In [ ]:
def student_extra(metrics):
    # Add a new diagnostic here; return {} keeps all demonstrations runnable.
    return {}


In [ ]:
# Numerical and callback checks: errors are not replaced by dummy results.
np.testing.assert_allclose(posterior.sum(),1)
np.testing.assert_allclose(messages['marginals'],ref['marginals'],atol=1e-12)
v=mean_field(model)
assert np.all(np.diff(v['elbo'])>=-1e-11)
assert np.isclose(v['elbo'][-1]+v['reverse_kl'],ref['log_z'])
for method in ['Exact','VI','MC','IS','MH','Gibbs']:
    plot, metrics, exported = static_experiment(method,n=100,extra=student_extra)
    assert 0<=metrics['estimate_probability']<=1 and np.isfinite(metrics['total_variation'])
    assert len(pd.read_csv(exported))==1
assert len(filter_table)==3 and np.isfinite(filter_table.particle_filter).all()
print('W6 numerical and app callback checks passed. Browser interaction is a separate check.')


### Web application

In Colab, the next cell launches the app automatically. Use **Compare with exact posterior**, change the method and stay probability, then export the CSV. Open **Sequential observations** for the optional filtering experiment. In a local notebook, set `LAUNCH_WEB_APP=True` to launch. The app requires its runtime to remain active.

Finish with an evidence-bounded claim: fixed data/model/query, settings and seeds, observed difference, and one limit of your conclusion.


In [ ]:
import os
IN_COLAB = bool(os.environ.get('COLAB_RELEASE_TAG')) or 'google.colab' in sys.modules
LAUNCH_WEB_APP = IN_COLAB
demo = create_app(extra=student_extra)
if LAUNCH_WEB_APP:
    demo.launch(share=IN_COLAB,inline=True,prevent_thread_lock=True)
else:
    print('Workbench built. Set LAUNCH_WEB_APP=True above and rerun this cell to open it locally.')
